In [1]:
import pandas as pd

In [2]:
patients = pd.read_csv("patients.csv")
consultations = pd.read_csv("consultations.csv")

patients.shape

(41, 6)

In [3]:
consultations.shape

(71, 6)

In [4]:
patients.columns

Index(['patient_id', 'nom', 'prenom', 'sexe', 'ville', 'date_naissance'], dtype='object')

In [5]:
consultations.columns

Index(['consultation_id', 'patient_id', 'date_consultation', 'medecin',
       'motif', 'cout_fcfa'],
      dtype='object')

### Valeurs manquantes

In [6]:
patients.isna().sum()

patient_id        0
nom               0
prenom            0
sexe              0
ville             2
date_naissance    0
dtype: int64

In [7]:
consultations.isna().sum()

consultation_id      0
patient_id           0
date_consultation    0
medecin              0
motif                0
cout_fcfa            3
dtype: int64

### Valeurs dupliquées

In [8]:
patients.duplicated().sum()

np.int64(1)

In [9]:
consultations.duplicated().sum()

np.int64(1)

In [10]:
patients = patients.drop_duplicates()
consultations = consultations.drop_duplicates()

In [11]:
patients.duplicated().sum()

np.int64(0)

In [12]:
consultations.duplicated().sum()

np.int64(0)

### Uniformiser la colonne 'ville' 

In [13]:
patients["ville"] = patients["ville"].str.lower().str.title()

# JOINTURE INTERNE (INNER JOIN)

In [22]:
patients.head()

,patient_id,nom,prenom,sexe,ville,date_naissance
0,1001,Nyaku,Ayite,M,Bassar,2015-05-07
1,1002,Boukpessi,Akosua,F,Sokodé,1978-03-22
2,1003,Foli,Komla,M,Kpalimé,1984-06-02
3,1004,Dogbe,Abla,F,NaN,2012-06-12
4,1005,Nyaku,Kossi,M,Sokodé,1996-12-16


In [23]:
consultations.head()

,consultation_id,patient_id,date_consultation,medecin,motif,cout_fcfa
0,2001,1001,2024-11-25,Dr. Amouzou,Consultation prénatale,5000.0
1,2002,1030,2024-10-03,Dr. Assih,Douleurs abdominales,NaN
2,2003,1002,2024-01-16,Dr. Assih,Contrôle diabète,5000.0
3,2004,1009,2024-09-03,Dr. Tchalla,Grippe,5000.0
4,2005,1033,2024-08-20,Dr. Tchalla,Paludisme,1000.0


In [24]:
merge_inner = pd.merge(consultations, patients, on="patient_id", how="inner")
merge_inner.shape

(68, 11)

Moins de lignes que consultations initial : les patient_id invalides (qui n'existent pas dans patients) sont exclus.

In [25]:
merge_inner.head()

,consultation_id,patient_id,date_consultation,medecin,motif,cout_fcfa,nom,prenom,sexe,ville,date_naissance
0,2001,1001,2024-11-25,Dr. Amouzou,Consultation prénatale,5000.0,Nyaku,Ayite,M,Bassar,2015-05-07
1,2002,1030,2024-10-03,Dr. Assih,Douleurs abdominales,NaN,Kpodar,Akosua,F,Dapaong,1991-01-19
2,2003,1002,2024-01-16,Dr. Assih,Contrôle diabète,5000.0,Boukpessi,Akosua,F,Sokodé,1978-03-22
3,2004,1009,2024-09-03,Dr. Tchalla,Grippe,5000.0,Dogbe,Komla,M,Kara,2014-07-12
4,2005,1033,2024-08-20,Dr. Tchalla,Paludisme,1000.0,Agbeko,Akpene,F,Sokodé,2010-01-27


# JOINTURE GAUCHE (LEFT JOIN)

In [26]:
merge_left = pd.merge(consultations, patients, on="patient_id", how="left")

In [36]:
lignes_orphelines = merge_left[merge_left["nom"].isna()]
print(lignes_orphelines)
print(len(lignes_orphelines))

    consultation_id  patient_id date_consultation      medecin  \
13             2014        1090        2024-05-24   Dr. Kolani   
59             2060        1090        2024-07-01  Dr. Tchalla   

                    motif  cout_fcfa  nom prenom sexe ville date_naissance  
13              Paludisme     2000.0  NaN    NaN  NaN   NaN            NaN  
59  Consultation générale     1500.0  NaN    NaN  NaN   NaN            NaN  
2


Il y a 2 patients aux id invlides.

# JOINTURE DROITE (RIGHT JOIN)

In [37]:
merge_right = pd.merge(consultations, patients, on="patient_id", how="right")

In [38]:
patients_sans_consultation = merge_right[merge_right["consultation_id"].isna()]
print(patients_sans_consultation[["patient_id", "nom", "prenom"]])
print(len(patients_sans_consultation))

    patient_id        nom  prenom
9         1006     Kponou    Edem
10        1007     Kponou    Sena
24        1013       Foli  Mawuli
32        1018  Boukpessi   Komla
34        1020      Dogbe  Delali
40        1024     Amegan   Adjoa
64        1034     Agbeko   Komla
75        1040     Adjovi     Ama
8


8 patients n'ont aucune consultation enregistrée.

# JOINTURE COMPLETE (OUTER JOIN)

In [39]:
merge_outer = pd.merge(consultations, patients, on="patient_id", how="outer")
print(merge_outer.shape)

(78, 11)


outer = inner + les orphelins de consultations + les patients sans consultation

### Analyse apres jointure

À partir du résultat de la jointure interne, calcule :

- a) le coût total des consultations par ville
- b) le nombre de consultations par médecin, trié du plus au moins actif
- c) le motif de consultation le plus fréquent par ville 

In [47]:
cout_total_inner = merge_inner.groupby('ville')['cout_fcfa'].sum()
cout_total_inner.sort_values(ascending=False)

ville
Sokodé      34000.0
Kara        29000.0
Bassar      20000.0
Kpalimé     18000.0
Dapaong     13000.0
Atakpamé    10500.0
Lomé         6500.0
Tsévié       3000.0
Name: cout_fcfa, dtype: float64

In [48]:
nbre_consultation_medecin_inner = merge_inner['medecin'].value_counts()
nbre_consultation_medecin_inner

medecin
Dr. Assih      15
Dr. Amouzou    14
Dr. Tchalla    13
Dr. Bakoma     13
Dr. Kolani     13
Name: count, dtype: int64

In [49]:
motif_consultation = merge_inner.groupby('ville')['motif'].value_counts()
motif_consultation

ville     motif                 
Atakpamé  Grippe                    2
          Contrôle diabète          1
          Paludisme                 1
          Vaccination               1
Bassar    Consultation prénatale    2
          Fièvre typhoïde           2
          Blessure                  1
          Consultation générale     1
          Contrôle diabète          1
          Grippe                    1
Dapaong   Douleurs abdominales      3
          Consultation prénatale    2
          Consultation générale     1
          Contrôle diabète          1
          Paludisme                 1
Kara      Contrôle diabète          3
          Consultation générale     2
          Grippe                    2
          Blessure                  1
          Douleurs abdominales      1
          Fièvre typhoïde           1
          Hypertension              1
Kpalimé   Contrôle diabète          2
          Paludisme                 2
          Blessure                  1
          Consult